# Eedi NeurIPS 2020 (Task 3 & 4) — Full 11-Step Multimodal Preprocessing on Google Colab (A100 / L4 / T4)

Notebook này thực thi **toàn bộ pipeline tiền xử lý dữ liệu đa phương thức 11 bước từ đầu đến cuối** cho bộ dữ liệu **Eedi NeurIPS 2020 (Task 3 & 4)** chỉ từ dữ liệu raw bạn đã lưu trên **Google Drive**.

### Kiến trúc & Đầu ra (Two-Tier Storage Architecture)
Sau khi hoàn tất, toàn bộ các tệp sẽ tương thích trực tiếp (zero code change) với `ACDMMKTv3` (Full & Ablations 1–3), `MGMMKT`, `MGMMKTv2`, và bộ đánh giá phân tầng `pyKT`:
- **Tier 1 (`data/processed/Eedi/`)**: `static_item_bank.pt`, `mg_static_item_bank.pt`, `acd_v3_aux_metadata.pt`
- **Tier 2 (`data/processed/Eedi/`)**: `sequences_fold_{0..4}.pt`, `sequences_test_public.pt`, `sequences_test_private.pt`, `sequences_test.pt`, `test_strata_masks.pt`, `metadata_summary.json`
- **Features & Splits**: `data/features/Eedi/` (`apple_vision_ocr.jsonl` sinh tự động bằng EasyOCR GPU nếu chạy mới trên Linux, `layout_inventory.json`, `rois/`, `stem_224/`, `ocr_texts_q.jsonl`, `solutions_s.jsonl`) và `data/splits/Eedi/`

### Checklist 11 Bước thực thi
1. **Step 1**: Kiểm toán 1-to-1 (`1,508,917` dòng), xác minh tách biệt UserId, cập nhật `CorrectAnswer` theo timestamp mới nhất (`[43, 84, 206, 860]`), hash MD5 loại trùng ảnh (`947` ảnh duy nhất), chuẩn hóa 85 KCs (`57 L3 + 25 L2 + 3 L1`), gom `50` cặp chuyên gia Task 3.
2. **Step 2**: Trích xuất OCR bounding boxes (dùng cache Apple Vision nếu có hoặc tự chạy **EasyOCR CUDA** trên A100), phân tích layout 4 badges A/B/C/D, cờ máy tính, cắt `RoI_stem` & `RoI_opt_A..D` sạch không nhãn.
3. **Step 3**: Kết hợp **Ollama `qwen2.5vl:3b` + Region-Aware OCR hints** trích xuất câu hỏi (kèm tiền tố `[DOMAIN | TOPIC | SKILL]`) & 4 đáp án dạng inline LaTeX `$...$` + `[BLANK]` + `[VISUAL_DIAGRAM_OPTION_k]` (không rò rỉ đáp án, checkpoint từng dòng JSONL).
4. **Step 4**: Dùng **`qwen2.5vl:3b`** (cho câu có hình) & **`qwen2.5:7b`** (cho câu thuần chữ) sinh lời giải sư phạm 3 phần neo chặt đáp án đúng (`solutions_s.jsonl`, tự kiểm chứng chống mâu thuẫn).
5. **Step 5**: Phân định lai CV + VLM (`m_img^q`) & letterbox `224x224` kèm mask patch `7x7` (`M_patch_q`).
6. **Step 6 & 7**: Trích xuất frozen embeddings bằng **`roberta-base`** & **`openai/clip-vit-base-patch16`** trên GPU + xây dựng siêu đồ thị phân cấp `H [85, 948]` & `G [85, 85]`.
7. **Step 8–11**: Sắp xếp 4 cấp thời gian, chia 5-fold phân tầng cân bằng câu hiếm (`min N_q >= 3`), Empiricial Bayes CTT & Dirichlet distractors chống rò rỉ theo từng fold, cửa sổ trượt giữ ngữ cảnh (`L=200, S=100`), xuất `.pt` và chạy kiểm thử **10 Acceptance Gates**.

> **Lưu ý trước khi chạy (`Runtime -> Change runtime type -> A100 GPU`):**  
> Hãy đảm bảo mã nguồn mới nhất (chứa `scripts/preprocess_eedi_task34.py` và `src/data/eedi_preprocessor.py`) đã được đẩy lên GitHub hoặc đặt trong Google Drive (`/content/drive/MyDrive/kt-mmkt-research`).


In [ ]:
# [1] Kiểm tra môi trường Google Colab & phần cứng GPU (A100 / L4 / T4)
import os
import shutil
import sys
import torch

is_colab = "google.colab" in sys.modules
print(f"Google Colab: {is_colab} | Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU CUDA detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU detected! Strongly recommend A100 GPU (Runtime -> Change runtime type) for Ollama & CLIP/RoBERTa speed.")

disk = shutil.disk_usage("/content" if os.path.exists("/content") else ".")
print(f"Local disk free: {disk.free / (1024 ** 3):.1f} GB")


In [ ]:
# [2] Kết nối Google Drive & Thiết lập thư mục sao lưu bền vững
from pathlib import Path

MOUNT_DRIVE = True
DRIVE_BACKUP_ROOT = Path("/content/drive/MyDrive/kt_mmkt_research_backup/Eedi")

if is_colab and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ("processed", "features", "splits"):
        (DRIVE_BACKUP_ROOT / sub).mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Backup root: {DRIVE_BACKUP_ROOT}")
else:
    DRIVE_BACKUP_ROOT = Path("./backup_eedi")
    print(f"Non-Colab local mode. Backup root: {DRIVE_BACKUP_ROOT}")


In [ ]:
# [3] Thiết lập Repository kt-mmkt-research trên Local NVMe SSD (/content/kt-mmkt-research)
import os
import shutil
import sys
from pathlib import Path

REPO_URL = "https://github.com/nbao0208/kt-mmkt-research.git"
WORK_DIR = Path("/content/kt-mmkt-research") if is_colab else Path.cwd()
DRIVE_REPO_CANDIDATES = [
    Path("/content/drive/MyDrive/kt-mmkt-research"),
    Path("/content/drive/MyDrive/graduate/kt-mmkt-research"),
]

if Path("scripts/preprocess_eedi_task34.py").exists():
    WORK_DIR = Path.cwd()
    print(f"Using current repository directory: {WORK_DIR}")
else:
    if WORK_DIR.exists():
        os.chdir(WORK_DIR)
        !git pull origin main || true
    else:
        !git clone {REPO_URL} {WORK_DIR}
        os.chdir(WORK_DIR)

    # Nếu GitHub chưa có commit mới nhất nhưng thư mục project trên Drive đã có code mới, đồng bộ các thư mục mã nguồn
    if not (WORK_DIR / "scripts/preprocess_eedi_task34.py").exists():
        for cand in DRIVE_REPO_CANDIDATES:
            if (cand / "scripts/preprocess_eedi_task34.py").exists():
                print(f"Syncing latest pipeline source code from Drive: {cand}")
                for folder in ("configs", "src", "scripts", "tests", "docs"):
                    src_f = cand / folder
                    dst_f = WORK_DIR / folder
                    if src_f.exists():
                        shutil.copytree(src_f, dst_f, dirs_exist_ok=True)
                break

if not (WORK_DIR / "scripts/preprocess_eedi_task34.py").exists():
    raise FileNotFoundError(
        "Missing scripts/preprocess_eedi_task34.py! Hãy git push code mới nhất lên GitHub "
        "hoặc đồng bộ thư mục kt-mmkt-research lên /content/drive/MyDrive/kt-mmkt-research."
    )

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
print(f"Active Working Directory: {os.getcwd()}")


In [ ]:
# [4] Cài đặt Dependencies & Kiểm tra thư viện (PyTorch, Transformers, EasyOCR)
!pip install -q pyyaml transformers easyocr pillow scipy pandas numpy pytest tqdm matplotlib scikit-learn

import easyocr
import pandas as pd
import scipy
import transformers

print(
    f"Dependencies OK | transformers={transformers.__version__} | "
    f"easyocr={easyocr.__version__} | pandas={pd.__version__} | cuda={torch.cuda.is_available()}"
)


In [ ]:
# [5] Tự động tìm kiếm toàn bộ Raw Dataset Eedi trên Google Drive & Copy về Local SSD
import glob
import shutil
import zipfile
from pathlib import Path

# Nếu bạn đặt raw dataset ở vị trí cụ thể và muốn chỉ định thẳng, điền vào biến dưới đây (để trống = tự động dò tìm)
RAW_EEDI_DRIVE_OVERRIDE = ""

TARGET_RAW_DATA = Path("data/raw/Eedi NeurIPS 2020/data_extracted/data")
TARGET_RAW_DATA.mkdir(parents=True, exist_ok=True)

def _has_required_raw_structure(base: Path) -> bool:
    return (
        (base / "train_data" / "train_task_3_4.csv").exists()
        and (base / "metadata" / "answer_metadata_task_3_4.csv").exists()
        and (base / "metadata" / "question_metadata_task_3_4.csv").exists()
        and (base / "metadata" / "student_metadata_task_3_4.csv").exists()
        and (base / "metadata" / "subject_metadata.csv").exists()
        and (base / "images" / "0.jpg").exists()
    )

if _has_required_raw_structure(TARGET_RAW_DATA):
    print(f"Raw Eedi data already prepared at: {TARGET_RAW_DATA}")
else:
    found_raw_root = None
    if RAW_EEDI_DRIVE_OVERRIDE:
        cand = Path(RAW_EEDI_DRIVE_OVERRIDE)
        if _has_required_raw_structure(cand):
            found_raw_root = cand

    if found_raw_root is None:
        common_candidates = [
            Path("/content/drive/MyDrive/Eedi NeurIPS 2020/data_extracted/data"),
            Path("/content/drive/MyDrive/Eedi NeurIPS 2020/data"),
            Path("/content/drive/MyDrive/Eedi NeurIPS 2020"),
            Path("/content/drive/MyDrive/Eedi/data_extracted/data"),
            Path("/content/drive/MyDrive/Eedi/data"),
            Path("/content/drive/MyDrive/Eedi"),
            Path("/content/drive/MyDrive/data/raw/Eedi NeurIPS 2020/data_extracted/data"),
        ]
        for cand in common_candidates:
            if _has_required_raw_structure(cand):
                found_raw_root = cand
                break

    if found_raw_root is None:
        matches = glob.glob("/content/drive/MyDrive/**/train_task_3_4.csv", recursive=True)
        if matches:
            cand = Path(matches[0]).parent.parent
            if (cand / "metadata" / "answer_metadata_task_3_4.csv").exists():
                found_raw_root = cand
                print(f"Found raw Eedi folder via recursive search: {found_raw_root}")

    # Fallback: tìm file ZIP nếu dataset raw trên Drive đang ở dạng nén
    if found_raw_root is None:
        zip_matches = glob.glob("/content/drive/MyDrive/**/*Eedi*.zip", recursive=True) + glob.glob(
            "/content/drive/MyDrive/**/data*.zip", recursive=True
        )
        for zm in zip_matches:
            zp = Path(zm)
            print(f"Checking zip archive on Drive: {zp}")
            extract_tmp = Path("/content/eedi_unzipped_tmp")
            extract_tmp.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(extract_tmp)
            sub_matches = list(extract_tmp.rglob("train_task_3_4.csv"))
            if sub_matches:
                found_raw_root = sub_matches[0].parent.parent
                print(f"Extracted raw Eedi data to {found_raw_root}")
                break

    if found_raw_root is None:
        raise FileNotFoundError(
            "Không tìm thấy dữ liệu raw Eedi (train_task_3_4.csv) trên Google Drive! "
            "Hãy kiểm tra tên thư mục trên MyDrive hoặc điền đường dẫn vào RAW_EEDI_DRIVE_OVERRIDE."
        )

    # Nếu images.zip tồn tại cạnh train_data mà chưa giải nén images/, giải nén trực tiếp về local SSD
    if not (found_raw_root / "images" / "0.jpg").exists():
        img_zips = list(found_raw_root.glob("images*.zip")) + list(found_raw_root.parent.glob("images*.zip"))
        if img_zips:
            print(f"Extracting {img_zips[0]} to local SSD...")
            with zipfile.ZipFile(img_zips[0], "r") as zf:
                zf.extractall(TARGET_RAW_DATA)

    print(f"Copying raw Eedi CSVs & images from {found_raw_root} to fast local SSD {TARGET_RAW_DATA}...")
    for sub in ("train_data", "test_data", "metadata", "images"):
        src_sub = found_raw_root / sub
        dst_sub = TARGET_RAW_DATA / sub
        if src_sub.exists() and not dst_sub.exists():
            shutil.copytree(src_sub, dst_sub)

# Xác nhận đầy đủ tệp raw bắt buộc trước khi chạy pipeline
required_files = [
    TARGET_RAW_DATA / "train_data" / "train_task_3_4.csv",
    TARGET_RAW_DATA / "test_data" / "test_public_task_4_more_splits.csv",
    TARGET_RAW_DATA / "test_data" / "test_private_task_4_more_splits.csv",
    TARGET_RAW_DATA / "test_data" / "quality_response_remapped_public.csv",
    TARGET_RAW_DATA / "test_data" / "quality_response_remapped_private.csv",
    TARGET_RAW_DATA / "metadata" / "answer_metadata_task_3_4.csv",
    TARGET_RAW_DATA / "metadata" / "question_metadata_task_3_4.csv",
    TARGET_RAW_DATA / "metadata" / "student_metadata_task_3_4.csv",
    TARGET_RAW_DATA / "metadata" / "subject_metadata.csv",
    TARGET_RAW_DATA / "images" / "0.jpg",
    TARGET_RAW_DATA / "images" / "947.jpg",
]
for rf in required_files:
    assert rf.exists(), f"Missing required raw Eedi file: {rf}"
print(f"Verified all required raw Eedi files & {len(list((TARGET_RAW_DATA / 'images').glob('*.jpg')))} slide images!")

# Khôi phục các checkpoint (nếu từng chạy dở trước đó và sao lưu trên Drive)
feat_dir = Path("data/features/Eedi")
splits_dir = Path("data/splits/Eedi")
feat_dir.mkdir(parents=True, exist_ok=True)
splits_dir.mkdir(parents=True, exist_ok=True)

for fname in ("apple_vision_ocr.jsonl", "layout_inventory.json", "ocr_texts_q.jsonl", "solutions_s.jsonl"):
    bak = DRIVE_BACKUP_ROOT / "features" / fname
    dst = feat_dir / fname
    if bak.exists() and not dst.exists():
        shutil.copy2(bak, dst)
        print(f"Restored checkpoint from Drive: {fname}")

split_bak = DRIVE_BACKUP_ROOT / "splits" / "train_val_folds.json"
if split_bak.exists() and not (splits_dir / "train_val_folds.json").exists():
    shutil.copy2(split_bak, splits_dir / "train_val_folds.json")
    print("Restored 5-fold split checkpoint from Drive: train_val_folds.json")


In [ ]:
# [6] Cài đặt & Khởi chạy Ollama GPU Server + Tải qwen2.5vl:3b & qwen2.5:7b + Pre-warm HuggingFace
import json
import os
import subprocess
import time
import urllib.request

# Cài Ollama Linux binary trên Colab
if shutil.which("ollama") is None:
    print("Installing Ollama on Colab...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
else:
    print("Ollama already installed:", shutil.which("ollama"))

# Khởi động Ollama server nền với Flash Attention & Keep-Alive 60m trên GPU A100
ollama_env = os.environ.copy()
ollama_env["OLLAMA_FLASH_ATTENTION"] = "1"
ollama_env["OLLAMA_KEEP_ALIVE"] = "60m"
ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"

def _ollama_ready() -> bool:
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not _ollama_ready():
    print("Starting background ollama serve...")
    with open("/tmp/ollama_serve.log", "w") as logf:
        subprocess.Popen(
            ["ollama", "serve"],
            env=ollama_env,
            stdout=logf,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    for _ in range(30):
        if _ollama_ready():
            break
        time.sleep(1)

assert _ollama_ready(), "Ollama server failed to start! Check /tmp/ollama_serve.log"
print("Ollama server is LIVE at http://127.0.0.1:11434")

# Kéo 2 local models cần cho Step 3 (VLM OCR) & Step 4 (Math Solution Recovery)
!ollama pull qwen2.5vl:3b
!ollama pull qwen2.5:7b
!ollama list

# Tải trước tokenizer + weights của RoBERTa-base và CLIP ViT-B/16 để Step 6 chạy mượt offline/CUDA
from transformers import AutoModel, AutoTokenizer, CLIPImageProcessor, CLIPVisionModel
print("Caching Hugging Face models (roberta-base, openai/clip-vit-base-patch16)...")
AutoTokenizer.from_pretrained("roberta-base")
AutoModel.from_pretrained("roberta-base")
CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch16")
CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch16")
print("All local Ollama & Hugging Face backbones ready!")


In [ ]:
# [7] Thực thi toàn bộ Pipeline 11 Bước từ đầu đến cuối + Auto-Sync Checkpoint lên Drive
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

# Đổi thành True CHỈ KHI bạn muốn xóa toàn bộ cache trung gian (OCR JSONL, layout, solutions) để chạy lại mới 100%
RESET_INTERMEDIATE_CACHE = False

if RESET_INTERMEDIATE_CACHE:
    for p in [
        Path("data/features/Eedi/apple_vision_ocr.jsonl"),
        Path("data/features/Eedi/layout_inventory.json"),
        Path("data/features/Eedi/ocr_texts_q.jsonl"),
        Path("data/features/Eedi/solutions_s.jsonl"),
        Path("data/splits/Eedi/train_val_folds.json"),
    ]:
        if p.exists():
            p.unlink()
    print("Cleared intermediate JSONL/JSON caches cleanly.")

stop_sync = threading.Event()

def _sync_features_to_drive():
    src_feat = Path("data/features/Eedi")
    dst_feat = DRIVE_BACKUP_ROOT / "features"
    dst_splits = DRIVE_BACKUP_ROOT / "splits"
    dst_feat.mkdir(parents=True, exist_ok=True)
    dst_splits.mkdir(parents=True, exist_ok=True)
    for fname in ("apple_vision_ocr.jsonl", "layout_inventory.json", "ocr_texts_q.jsonl", "solutions_s.jsonl"):
        sp = src_feat / fname
        if sp.exists():
            try:
                shutil.copy2(sp, dst_feat / fname)
            except Exception as exc:
                print(f"[Backup warn] {fname}: {exc}")
    fold_p = Path("data/splits/Eedi/train_val_folds.json")
    if fold_p.exists():
        try:
            shutil.copy2(fold_p, dst_splits / "train_val_folds.json")
        except Exception:
            pass

def _periodic_backup_loop():
    while not stop_sync.wait(timeout=90):
        _sync_features_to_drive()

sync_thread = threading.Thread(target=_periodic_backup_loop, daemon=True)
sync_thread.start()

t_start = time.time()
print("Launching full 11-step Eedi Task 3 & 4 preprocessing pipeline...")
proc = subprocess.Popen(
    [sys.executable, "-u", "scripts/preprocess_eedi_task34.py", "--config", "configs/dataset/eedi_task34.yaml"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()

stop_sync.set()
sync_thread.join(timeout=5)
_sync_features_to_drive()

elapsed = time.time() - t_start
if ret != 0:
    raise RuntimeError(f"Preprocessing pipeline exited with non-zero status {ret} after {elapsed/60:.1f} min.")
print(f"\nPipeline 11 bước hoàn tất thành công sau {elapsed/60:.1f} phút!")


In [ ]:
# [8] Kiểm tra thống kê thực nghiệm đầu ra & Trực quan hóa mẫu RoI / OCR / Lời giải
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch

proc_dir = Path("data/processed/Eedi")
with open(proc_dir / "metadata_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print("=== METADATA SUMMARY (Eedi Task 3 & 4) ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\n=== GENERATED ARTIFACT FILES & SIZES ===")
for pt_file in sorted(proc_dir.glob("*")):
    size_mb = pt_file.stat().st_size / (1024 ** 2)
    print(f"  {pt_file.name:<32} {size_mb:8.2f} MB")

# Đọc mẫu OCR + Solution + RoI
ocr_sample = {}
with open("data/features/Eedi/ocr_texts_q.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec["qid"] in (1, 10, 55, 205):
            ocr_sample[rec["qid"]] = rec

sol_sample = {}
with open("data/features/Eedi/solutions_s.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec["qid"] in (1, 10, 55, 205):
            sol_sample[rec["qid"]] = rec

for q in (1, 10, 55, 205):
    print(f"\n--- Sample Q{q} | m_img^q={ocr_sample[q]['has_stem_diagram']} ---")
    print("  T_q :", ocr_sample[q]["text_q"])
    print("  T_s :", sol_sample[q]["text_s"])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, q in enumerate((1, 10, 55, 205)):
    axes[0, col].imshow(Image.open(f"data/features/Eedi/rois/{q}_stem.png"))
    axes[0, col].set_title(f"Q{q} RoI_stem (vis={ocr_sample[q]['has_stem_diagram']})")
    axes[0, col].axis("off")
    axes[1, col].imshow(Image.open(f"data/features/Eedi/stem_224/{q}.png"))
    axes[1, col].set_title(f"Q{q} Letterbox 224x224")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# [9] Chạy kiểm thử toàn bộ 10 Acceptance Gates + Tương thích Mô hình & Ablations
from pathlib import Path
import subprocess
import sys

test_cmd = [
    sys.executable, "-m", "pytest",
    "tests/test_eedi_task34_preprocessing.py",
    "tests/test_acd_mmkt_v3_model.py",
    "tests/test_acd_mmkt_v3_ablation_study.py",
    "tests/test_explainability.py",
]
if Path("data/raw/XES3G5M").exists() and Path("data/processed/XES3G5M").exists():
    test_cmd.append("tests/test_xes3g5m_preprocessing.py")
test_cmd.append("-v")
print("Running:", " ".join(test_cmd))
res = subprocess.run(test_cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Một số kiểm thử chưa đạt! Vui lòng kiểm tra log ở trên.")
print("ALL 10 ACCEPTANCE GATES & REGRESSION TESTS PASSED 100%!")


In [ ]:
# [10] Sao lưu toàn bộ đầu ra cuối cùng lên Google Drive & Đóng gói ZIP tùy chọn
import shutil
import zipfile
from pathlib import Path

AUTO_DOWNLOAD_ZIP = False  # Đổi thành True nếu muốn trình duyệt Colab tự tải tệp ZIP về máy

for folder_name in ("processed", "splits"):
    src = Path("data") / folder_name / "Eedi"
    dst = DRIVE_BACKUP_ROOT / folder_name
    if src.exists():
        print(f"Syncing {src} -> {dst}...")
        shutil.copytree(src, dst, dirs_exist_ok=True)

# Sao lưu toàn bộ features trung gian (.jsonl/.json + ảnh stem_224/)
feat_src = Path("data/features/Eedi")
feat_dst = DRIVE_BACKUP_ROOT / "features"
feat_dst.mkdir(parents=True, exist_ok=True)
for item in feat_src.iterdir():
    if item.is_file():
        shutil.copy2(item, feat_dst / item.name)
shutil.copytree(feat_src / "stem_224", feat_dst / "stem_224", dirs_exist_ok=True)

zip_path = DRIVE_BACKUP_ROOT / "eedi_task34_processed_bundle.zip"
print(f"Creating portable ZIP archive at {zip_path}...")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for base_dir in (Path("data/processed/Eedi"), Path("data/splits/Eedi")):
        for fpath in sorted(base_dir.rglob("*")):
            if fpath.is_file():
                zf.write(fpath, arcname=str(fpath))
    for fname in ("apple_vision_ocr.jsonl", "layout_inventory.json", "ocr_texts_q.jsonl", "solutions_s.jsonl"):
        fp = feat_src / fname
        if fp.exists():
            zf.write(fp, arcname=str(fp))

print(f"Saved ZIP bundle ({zip_path.stat().st_size / (1024 ** 2):.1f} MB) at: {zip_path}")
if is_colab and AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
